# Treinamento Python — Nível 9 — Data Engineering com pandas

> **Data Engineering Track** | Python do zero ao Data Engineering
> Cada seção termina com um **exercício de fixação** e ao final há um **desafio integrador**.

---
# 📘 Aula 01 Pandas Basico

## NÍVEL 9 — Data Engineering | Aula 1: pandas Básico

pandas é a biblioteca mais usada para manipulação de dados em Python
Instalar: pip install pandas

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/home/lg/Documents/personal/projects/treinamento/python/nivel_9/dados")
BASE.mkdir(parents=True, exist_ok=True)

## 1. Criando DataFrames

DataFrame = tabela de dados (linhas x colunas)

A partir de lista de dicionários

In [ ]:
df = pd.DataFrame([
    {"nome": "Ana",   "depto": "Engenharia", "salario": 9500, "anos": 3},
    {"nome": "Bruno", "depto": "Analytics",  "salario": 6200, "anos": 2},
    {"nome": "Carla", "depto": "Engenharia", "salario": 4800, "anos": 1},
    {"nome": "Diego", "depto": "Analytics",  "salario": 3200, "anos": 0},
    {"nome": "Elena", "depto": "Engenharia", "salario": 12000,"anos": 7},
    {"nome": "Fábio", "depto": "Analytics",  "salario": 7800, "anos": 4},
])

print("=== DataFrame ===")
print(df)
print(f"\nShape: {df.shape}")     # (linhas, colunas)
print(f"Colunas: {list(df.columns)}")
print(f"\nTipos:\n{df.dtypes}")

## 2. Inspeção de dados

In [ ]:
print("\n=== Primeiras linhas ===")
print(df.head(3))

print("\n=== Estatísticas ===")
print(df.describe())

print("\n=== Info ===")
df.info()

## 3. Seleção de dados

Coluna única → Series

In [ ]:
print("\nNomes:", df["nome"].tolist())

# Múltiplas colunas → DataFrame
print(df[["nome", "salario"]])

# Filtros (como WHERE no SQL)
senior = df[df["salario"] > 8000]
print("\nSênior (>8k):")
print(senior[["nome", "salario"]])

# Múltiplos filtros
engenheiros_senior = df[(df["depto"] == "Engenharia") & (df["salario"] > 5000)]
print("\nEngenheiros sênior:")
print(engenheiros_senior[["nome", "depto", "salario"]])

# loc (por label) e iloc (por posição)
print(df.loc[0])          # primeira linha por índice
print(df.iloc[-1])        # última linha por posição
print(df.loc[1:3, ["nome", "salario"]])  # linhas 1-3, colunas selecionadas

## 4. Manipulação de colunas

Nova coluna calculada

In [ ]:
df["bonus"] = df["salario"] * 0.10
df["total"] = df["salario"] + df["bonus"]
df["nivel"] = df["salario"].apply(
    lambda s: "Sênior" if s > 8000 else ("Pleno" if s >= 4000 else "Júnior")
)

print("\nCom colunas calculadas:")
print(df[["nome", "salario", "bonus", "total", "nivel"]])

# Renomear colunas
df = df.rename(columns={"anos": "anos_empresa"})

# Remover colunas
df_limpo = df.drop(columns=["bonus"])

## 5. Ordenação

In [ ]:
print("\nOrdenado por salário (desc):")
print(df.sort_values("salario", ascending=False)[["nome", "salario", "nivel"]])

print("\nOrdenado por depto e salário:")
print(df.sort_values(["depto", "salario"], ascending=[True, False])[["nome", "depto", "salario"]])

## 6. Salvando e carregando

In [ ]:
csv_path = BASE / "funcionarios.csv"
df.to_csv(csv_path, index=False)
df_lido = pd.read_csv(csv_path)
print(f"\nSalvo e relido: {len(df_lido)} registros")

## EXERCÍCIO DE FIXAÇÃO 9.1

Com o DataFrame de funcionários:
  1. Filtre quem tem mais de 2 anos na empresa
  2. Calcule um bônus de 5% por ano de empresa
  3. Adicione coluna "salario_ajustado" = salario + bônus calculado
  4. Ordene por salario_ajustado decrescente
  5. Imprima o top 3

In [ ]:
# Escreva seu código aqui


---
# 📘 Aula 02 Pandas Avancado

## NÍVEL 9 — Data Engineering | Aula 2: pandas Avançado

In [ ]:
import pandas as pd
import numpy as np

## 1. Limpeza de dados

Dados sujos (como vêm da vida real)

In [ ]:
df = pd.DataFrame({
    "id":       [1, 2, 3, 4, 5, 2, 6],
    "nome":     ["  Ana Lima", "BRUNO MELO  ", None, "Diego Neto", "  ELENA  ", "BRUNO MELO  ", "Fábio"],
    "idade":    [28, 35, None, 42, 29, 35, None],
    "salario":  [9500, 6200, 4800, None, 11000, 6200, 7000],
    "depto":    ["Engenharia", "analytics", "Engenharia", "Analytics", "engenharia", "analytics", "Analytics"],
    "email":    ["ana@e.com", "bruno@e.com", "invalido", "diego@e.com", "elena@e.com", "bruno@e.com", ""],
})

print("=== Dados brutos ===")
print(df)
print(f"\nNulos por coluna:\n{df.isnull().sum()}")

# Removendo duplicatas
df = df.drop_duplicates(subset=["id"])
print(f"\nApós drop_duplicates: {len(df)} linhas")

# Tratando nulos
df["idade"] = df["idade"].fillna(df["idade"].median())   # preenche com mediana
df["salario"] = df["salario"].fillna(df["salario"].mean())  # média
df = df.dropna(subset=["nome"])  # remove linhas onde nome é nulo

# Normalizando strings
df["nome"] = df["nome"].str.strip().str.title()
df["depto"] = df["depto"].str.strip().str.title()

# Validando emails
df["email_valido"] = df["email"].str.contains("@", na=False) & (df["email"] != "")

print("\n=== Dados limpos ===")
print(df.to_string(index=False))

## 2. groupby — agregações

In [ ]:
print("\n=== groupby ===")

# Soma, média, count por departamento
resumo = df.groupby("depto")["salario"].agg(["sum", "mean", "count", "min", "max"])
resumo.columns = ["total", "media", "headcount", "minimo", "maximo"]
print(resumo.round(2))

# Múltiplas colunas de agregação
stats = df.groupby("depto").agg(
    headcount=("id", "count"),
    salario_medio=("salario", "mean"),
    salario_total=("salario", "sum"),
    idade_media=("idade", "mean")
).round(2)
print("\nStats por depto:")
print(stats)

## 3. merge — joins entre DataFrames (como SQL)

In [ ]:
funcionarios = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "nome": ["Ana", "Bruno", "Carla", "Diego"],
    "depto_id": [10, 20, 10, 30]
})

departamentos = pd.DataFrame({
    "depto_id": [10, 20, 30, 40],
    "depto_nome": ["Engenharia", "Analytics", "RH", "Financeiro"],
    "budget": [500000, 300000, 150000, 400000]
})

# INNER JOIN (padrão)
inner = pd.merge(funcionarios, departamentos, on="depto_id")
print("\nINNER JOIN:")
print(inner)

# LEFT JOIN
left = pd.merge(funcionarios, departamentos, on="depto_id", how="left")
print("\nLEFT JOIN:")
print(left)

## 4. pivot_table — tabela dinâmica

In [ ]:
vendas = pd.DataFrame({
    "data":     ["2026-05-01","2026-05-01","2026-05-02","2026-05-02","2026-05-03"],
    "regiao":   ["Sul","Norte","Sul","Norte","Sul"],
    "produto":  ["Arroz","Feijão","Milho","Arroz","Feijão"],
    "valor":    [500, 300, 200, 450, 350]
})

pivot = pd.pivot_table(
    vendas,
    values="valor",
    index="regiao",
    columns="produto",
    aggfunc="sum",
    fill_value=0
)
print("\nPivot table:")
print(pivot)

## 5. Transformações avançadas

In [ ]:
df_v = vendas.copy()

# cumsum — soma acumulada
df_v = df_v.sort_values("data")
df_v["valor_acumulado"] = df_v.groupby("regiao")["valor"].cumsum()

# rank — posição relativa
df_v["rank_valor"] = df_v["valor"].rank(ascending=False).astype(int)

# cut — discretização em faixas
df["faixa_salarial"] = pd.cut(
    df["salario"],
    bins=[0, 5000, 9000, float("inf")],
    labels=["Júnior", "Pleno", "Sênior"]
)
print("\nFaixas salariais:")
print(df[["nome", "salario", "faixa_salarial"]].to_string(index=False))

## EXERCÍCIO DE FIXAÇÃO 9.2

Crie um DataFrame com 10 transações financeiras
(id, data, cliente, valor, tipo: "crédito"/"débito", status: "aprovado"/"cancelado")
Use pandas para:
  1. Filtrar apenas aprovadas
  2. Calcular total por tipo
  3. Calcular saldo (créditos - débitos)
  4. Gerar pivot com total por cliente x tipo

In [ ]:
# Escreva seu código aqui


---
# 🏆 Desafio Nivel 9

## NÍVEL 9 — DESAFIO FINAL (E DESAFIO GERAL DO TREINAMENTO) | Pipeline ETL Completo com pandas

CONTEXTO:
Você é um Data Engineer. Recebeu dados brutos de vendas de
múltiplas regiões e precisa construir um pipeline ETL completo
que aplica TUDO que foi aprendido no treinamento:
  - OOP (classes e herança)
  - Decorators e context managers
  - Tratamento de erros
  - Módulos e arquivos
  - pandas para análise
  - Type hints

In [ ]:
# Escreva seu código aqui


## CONFIGURAÇÃO

In [ ]:
BASE = Path("/home/lg/Documents/personal/projects/treinamento/python/nivel_9/dados")
BASE.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(BASE / "etl_final.log", encoding="utf-8", mode="w"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("etl_final")

## DECORATORS

In [ ]:
def log_etapa(funcao: Callable) -> Callable:
    @functools.wraps(funcao)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        logger.info(f"INICIANDO: {funcao.__name__}")
        try:
            resultado = funcao(*args, **kwargs)
            logger.info(f"CONCLUÍDO: {funcao.__name__} | {time.perf_counter()-t0:.4f}s")
            return resultado
        except Exception as e:
            logger.error(f"FALHOU: {funcao.__name__} | {type(e).__name__}: {e}")
            raise
    return wrapper


@contextmanager
def fase_pipeline(nome: str):
    logger.info(f"{'='*40}")
    logger.info(f"FASE: {nome}")
    t0 = time.perf_counter()
    yield
    logger.info(f"FIM FASE: {nome} | {time.perf_counter()-t0:.4f}s")

## EXCEÇÕES

In [ ]:
class ETLError(Exception): pass
class DadosInsuficientes(ETLError): pass
class SchemaError(ETLError): pass

## DATACLASSES DE CONFIGURAÇÃO

In [ ]:
@dataclass
class ConfigETL:
    nome: str
    colunas_obrigatorias: list[str] = field(default_factory=list)
    valor_minimo: float = 0.0
    taxa_imposto: float = 0.12
    taxa_desconto_premium: float = 0.08

## EXTRACT

In [ ]:
@log_etapa
def extrair_dados() -> pd.DataFrame:
    """Simula extração de múltiplas fontes."""

    # Fonte 1: vendas operacionais
    vendas = [
        {"id": "V001", "data": "2026-05-01", "regiao": "Sul",    "vendedor": "Ana Lima",    "produto": "Notebook",  "categoria": "Eletrônico", "qtd": 2,  "preco": 3500.00, "cliente_premium": True},
        {"id": "V002", "data": "2026-05-01", "regiao": "Norte",  "vendedor": "Bruno Melo",  "produto": "Cadeira",   "categoria": "Móvel",      "qtd": 5,  "preco": 750.00,  "cliente_premium": False},
        {"id": "V003", "data": "2026-05-01", "regiao": "Leste",  "vendedor": "Carla Souza", "produto": "Monitor",   "categoria": "Eletrônico", "qtd": 1,  "preco": 1800.00, "cliente_premium": True},
        {"id": "V004", "data": "2026-05-02", "regiao": "Sul",    "vendedor": "Diego Neto",  "produto": "Mouse",     "categoria": "Eletrônico", "qtd": 10, "preco": 89.00,   "cliente_premium": False},
        {"id": "V005", "data": "2026-05-02", "regiao": "Norte",  "vendedor": "Elena Rocha", "produto": "Notebook",  "categoria": "Eletrônico", "qtd": 1,  "preco": 4200.00, "cliente_premium": True},
        {"id": "V006", "data": "2026-05-02", "regiao": "Leste",  "vendedor": "Fábio Assis", "produto": "Mesa",      "categoria": "Móvel",      "qtd": 3,  "preco": 1200.00, "cliente_premium": False},
        {"id": "V007", "data": "2026-05-03", "regiao": "Sul",    "vendedor": "Gabi Costa",  "produto": "Monitor",   "categoria": "Eletrônico", "qtd": 2,  "preco": 1800.00, "cliente_premium": True},
        {"id": "V008", "data": "2026-05-03", "regiao": "Norte",  "vendedor": "Hugo Lima",   "produto": "Cadeira",   "categoria": "Móvel",      "qtd": 8,  "preco": 750.00,  "cliente_premium": False},
        {"id": "V009", "data": "2026-05-03", "regiao": "Leste",  "vendedor": "Íris Melo",   "produto": "Teclado",   "categoria": "Eletrônico", "qtd": 15, "preco": 250.00,  "cliente_premium": False},
        {"id": "V010", "data": "2026-05-03", "regiao": "Sul",    "vendedor": "João Assis",  "produto": "Mouse",     "categoria": "Eletrônico", "qtd": None,"preco": 89.00,  "cliente_premium": False},  # qtd nulo
        {"id": "V011", "data": "2026-05-03", "regiao": "Norte",  "vendedor": "Karen Dias",  "produto": "Mesa",      "categoria": "Móvel",      "qtd": 2,  "preco": -100.00, "cliente_premium": True},   # preço inválido
        {"id": "V003", "data": "2026-05-01", "regiao": "Leste",  "vendedor": "Carla Souza", "produto": "Monitor",   "categoria": "Eletrônico", "qtd": 1,  "preco": 1800.00, "cliente_premium": True},   # duplicata
    ]
    df = pd.DataFrame(vendas)
    logger.info(f"Extraídos {len(df)} registros brutos")
    return df

## TRANSFORM

In [ ]:
@log_etapa
def transformar(df: pd.DataFrame, config: ConfigETL) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Limpa, valida e enriquece os dados. Retorna (válidos, inválidos)."""

    # 1. Deduplicação
    antes = len(df)
    df = df.drop_duplicates(subset=["id"])
    logger.info(f"Deduplicação: {antes - len(df)} registros removidos")

    # 2. Separar inválidos
    mask_qtd_nula = df["qtd"].isnull()
    mask_preco_inv = df["preco"] <= config.valor_minimo

    inválidos = df[mask_qtd_nula | mask_preco_inv].copy()
    inválidos["motivo_rejeicao"] = ""
    inválidos.loc[mask_qtd_nula, "motivo_rejeicao"] += "qtd_nula "
    inválidos.loc[mask_preco_inv, "motivo_rejeicao"] += "preco_invalido "

    df = df[~(mask_qtd_nula | mask_preco_inv)].copy()

    if len(df) < 1:
        raise DadosInsuficientes("Nenhum registro válido após limpeza")

    logger.info(f"Registros válidos: {len(df)} | Inválidos: {len(inválidos)}")

    # 3. Tipagem
    df["qtd"] = df["qtd"].astype(int)
    df["data"] = pd.to_datetime(df["data"])

    # 4. Colunas calculadas
    df["subtotal"] = df["preco"] * df["qtd"]
    df["imposto"] = (df["subtotal"] * config.taxa_imposto).round(2)
    df["desconto"] = df.apply(
        lambda r: round(r["subtotal"] * config.taxa_desconto_premium, 2) if r["cliente_premium"] else 0,
        axis=1
    )
    df["total_final"] = (df["subtotal"] + df["imposto"] - df["desconto"]).round(2)
    df["faixa_valor"] = pd.cut(
        df["total_final"],
        bins=[0, 500, 2000, 5000, float("inf")],
        labels=["Baixo", "Médio", "Alto", "Premium"]
    )

    return df, inválidos

## ANALYZE

In [ ]:
@log_etapa
def analisar(df: pd.DataFrame) -> dict[str, Any]:
    por_regiao = df.groupby("regiao")["total_final"].agg(
        receita_total="sum", ticket_medio="mean", n_vendas="count"
    ).round(2)

    por_categoria = df.groupby("categoria")["total_final"].agg(
        receita_total="sum", n_vendas="count"
    ).round(2)

    top_produtos = df.groupby("produto")["total_final"].sum().sort_values(ascending=False).head(3)

    pivot = pd.pivot_table(df, values="total_final", index="regiao",
                           columns="categoria", aggfunc="sum", fill_value=0).round(2)

    return {
        "por_regiao": por_regiao,
        "por_categoria": por_categoria,
        "top_produtos": top_produtos,
        "pivot_regiao_categoria": pivot
    }

## LOAD

In [ ]:
@log_etapa
def salvar(df: pd.DataFrame, invalidos: pd.DataFrame, analises: dict, config: ConfigETL):
    df.to_csv(BASE / "vendas_processadas.csv", index=False)
    invalidos.to_csv(BASE / "vendas_rejeitadas.csv", index=False)

    resumo = {
        "pipeline": config.nome,
        "executado_em": datetime.now().isoformat(),
        "total_processados": len(df),
        "total_rejeitados": len(invalidos),
        "receita_total": round(df["total_final"].sum(), 2),
        "receita_por_regiao": analises["por_regiao"]["receita_total"].to_dict(),
    }
    with open(BASE / "resumo.json", "w", encoding="utf-8") as f:
        json.dump(resumo, f, indent=2, ensure_ascii=False)

    logger.info(f"Arquivos salvos em {BASE}")

## RELATÓRIO FINAL

In [ ]:
def relatorio_final(df: pd.DataFrame, invalidos: pd.DataFrame, analises: dict):
    print(f"\n{'='*60}")
    print(f"  RELATÓRIO FINAL — PIPELINE ETL DE VENDAS")
    print(f"{'='*60}")
    print(f"  Registros processados : {len(df)}")
    print(f"  Registros rejeitados  : {len(invalidos)}")
    print(f"  Receita total         : R${df['total_final'].sum():>12,.2f}")

    print(f"\n  Por Região:")
    print(analises["por_regiao"].to_string())

    print(f"\n  Por Categoria:")
    print(analises["por_categoria"].to_string())

    print(f"\n  Top 3 Produtos:")
    for prod, val in analises["top_produtos"].items():
        print(f"    {prod:<15} R${val:>10,.2f}")

    print(f"\n  Pivot Região x Categoria:")
    print(analises["pivot_regiao_categoria"].to_string())

    print(f"\n  Rejeitados:")
    if len(invalidos) > 0:
        print(invalidos[["id", "produto", "motivo_rejeicao"]].to_string(index=False))

    print(f"{'='*60}\n")

## EXECUÇÃO

In [ ]:
config = ConfigETL(
    nome="ETL Vendas Mensais",
    colunas_obrigatorias=["id", "produto", "qtd", "preco"],
    taxa_imposto=0.12,
    taxa_desconto_premium=0.08
)

try:
    with fase_pipeline("EXTRACT"):
        df_bruto = extrair_dados()

    with fase_pipeline("TRANSFORM"):
        df_limpo, df_rejeitados = transformar(df_bruto, config)

    with fase_pipeline("ANALYZE"):
        analises = analisar(df_limpo)

    with fase_pipeline("LOAD"):
        salvar(df_limpo, df_rejeitados, analises, config)

    relatorio_final(df_limpo, df_rejeitados, analises)

except DadosInsuficientes as e:
    logger.critical(f"Pipeline abortado: {e}")
except Exception as e:
    logger.critical(f"Erro fatal: {type(e).__name__}: {e}")
    raise